In [1]:
pip install streamlit plotly

Note: you may need to restart the kernel to use updated packages.


In [3]:
import streamlit as st
import pandas as pd
import joblib
import plotly.express as px
import os

# ==============================================================================
# 1. CONFIGURACIÓ DE LA PÀGINA
# ==============================================================================
st.set_page_config(page_title="Simulador Vulnerabilitat BCN", layout="wide", page_icon="🌍")

st.title("🌍 Simulador de Vulnerabilitat Energètica de Barcelona")
st.markdown("""
Aquesta eina permet avaluar l'impacte de diferents escenaris climàtics i demogràfics 
sobre l'Índex de Vulnerabilitat (CVI) utilitzant el model d'Intel·ligència Artificial (Random Forest) desenvolupat al TFG.
""")

# ==============================================================================
# 2. DICCIONARI D'ESCENARIS (Sincronitzat amb el teu codi d'entrenament)
# ==============================================================================
DICCIONARI_ESCENARIS = {
    '0': [
        'renda_disponible_Import_Euros', 'pad_dom_Llars_1_Avi_Sol', 
        'edificacions_any_Any_Mitja_Ponderat', 'heating_thermal_demand_intensity__2025-03-06', 
        'torrid_nights__2024-01-01'
    ],
    '1': [
        'pad_dom_Llars_1_Avi_Sol', 'edificacions_any_Any_Mitja_Ponderat', 
        'heating_thermal_demand_intensity__2025-03-06', 'torrid_nights__2024-01-01'
    ],
    '3A': [
        'renda_disponible_Import_Euros', 'pad_dom_Llars_1_Avi_Sol', 
        'edificacions_any_Any_Mitja_Ponderat', 'heating_degree_days__2024-01-01', 
        'heating_thermal_demand_intensity__2025-03-06'
    ],
    '3B': [
        'renda_disponible_Import_Euros', 'pad_dom_Llars_1_Avi_Sol', 
        'edificacions_any_Any_Mitja_Ponderat', 'torrid_nights__2024-01-01', 
        'vegetation_index_avg__2022-01-01'
    ],
    '4A': [
        'edificacions_any_Any_Mitja_Ponderat', 'edificacions_superficie_Superficie_m2', 
        'percentage_population_over_65__2022-01-01', 'percentage_single_person_households__2022-01-01', 
        'heating_thermal_demand_intensity__2025-03-06', 'torrid_nights__2024-01-01', 
        'vegetation_index_avg__2022-01-01'
    ]
}

opcions_menu = {
    "Escenari 0: Baseline (Amb Renda)": "0",
    "Escenari 1: Estudi d'Ablació (Sense Renda)": "1",
    "Escenari 3A: Model d'Hivern": "3A",
    "Escenari 3B: Model d'Estiu": "3B",
    "Escenari 4A: Predictor Urbà Complet": "4A"
}

# ==============================================================================
# 3. PANELL LATERAL: SELECCIÓ I CONTROLS
# ==============================================================================
st.sidebar.header("⚙️ Configuració del Model")
nom_escenari = st.sidebar.selectbox("📊 Selecciona l'Escenari a visualitzar:", list(opcions_menu.keys()))
escenari_actual = opcions_menu[nom_escenari]
columnes_X = DICCIONARI_ESCENARIS[escenari_actual]

st.sidebar.markdown("---")
st.sidebar.header("🛠️ Simulació (What-If)")
st.sidebar.markdown("Modifica els paràmetres urbans per veure com reacciona la ciutat en aquest escenari:")

# Només mostrem els sliders si la variable corresponent existeix a l'escenari triat
mod_renda = st.sidebar.slider("💶 Variació de la Renda (%)", -30, 30, 0, step=5) if 'renda_disponible_Import_Euros' in columnes_X else 0
mod_calor = st.sidebar.slider("🌡️ Increment Nits Tòrrides (%)", 0, 100, 0, step=5) if 'torrid_nights__2024-01-01' in columnes_X else 0
mod_avis_sols = st.sidebar.slider("🧓 Augment Avis Sols (%)", 0, 50, 0, step=5) if 'pad_dom_Llars_1_Avi_Sol' in columnes_X else 0
mod_gent_gran = st.sidebar.slider("👴 Augment Densitat Gent Gran (%)", 0, 50, 0, step=5) if 'percentage_population_over_65__2022-01-01' in columnes_X else 0
mod_verd = st.sidebar.slider("🌳 Augment del Verd Urbà (%)", 0, 50, 0, step=5) if 'vegetation_index_avg__2022-01-01' in columnes_X else 0

# ==============================================================================
# 4. FUNCIONS DE CÀRREGA DE DADES
# ==============================================================================
@st.cache_resource
def carregar_model(escenari):
    ruta = f'../models/model_vulnerabilitat_RF_Escenari_{escenari}.joblib'
    if os.path.exists(ruta):
        return joblib.load(ruta)
    return None

@st.cache_data
def carregar_dades(escenari):
    ruta = f'../data/processed/mapa_prediccions_Escenari_{escenari}.csv'
    if os.path.exists(ruta):
        return pd.read_csv(ruta, sep=';', decimal=',')
    return None

# ==============================================================================
# 5. EXECUCIÓ DEL CÀLCUL
# ==============================================================================
model = carregar_model(escenari_actual)
df_base = carregar_dades(escenari_actual)

if model is None or df_base is None:
    st.warning(f"⚠️ **Dades no trobades per a l'{nom_escenari}.**\nSi us plau, executa primer el teu script de Python d'entrenament posant `ESCENARI_ACTUAL = '{escenari_actual}'` per generar els arxius necessaris.")
else:
    # Fem una còpia per aplicar-hi les variacions
    df_simulat = df_base.copy()

    # Apliquem matemàticament les variacions només si la variable està a l'escenari
    if 'renda_disponible_Import_Euros' in columnes_X:
        df_simulat['renda_disponible_Import_Euros'] *= (1 + (mod_renda / 100))
    if 'torrid_nights__2024-01-01' in columnes_X:
        df_simulat['torrid_nights__2024-01-01'] *= (1 + (mod_calor / 100))
    if 'pad_dom_Llars_1_Avi_Sol' in columnes_X:
        df_simulat['pad_dom_Llars_1_Avi_Sol'] *= (1 + (mod_avis_sols / 100))
    if 'percentage_population_over_65__2022-01-01' in columnes_X:
        df_simulat['percentage_population_over_65__2022-01-01'] *= (1 + (mod_gent_gran / 100))
    if 'vegetation_index_avg__2022-01-01' in columnes_X:
        df_simulat['vegetation_index_avg__2022-01-01'] *= (1 + (mod_verd / 100))

    # Tornem a predir amb el Random Forest
    df_simulat['CVI_Simulat'] = model.predict(df_simulat[columnes_X])

    # ==============================================================================
    # 6. VISUALITZACIÓ (MÈTRIQUES I MAPA)
    # ==============================================================================
    col1, col2, col3 = st.columns(3)
    cvi_mitja_base = df_base['CVI_Predit_RF'].mean()
    cvi_mitja_simulat = df_simulat['CVI_Simulat'].mean()
    diferencia = cvi_mitja_simulat - cvi_mitja_base

    col1.metric(label="CVI Mitjà (Sense alteracions)", value=f"{cvi_mitja_base:.2f}")
    col2.metric(label="CVI Mitjà (Simulat)", value=f"{cvi_mitja_simulat:.2f}", delta=f"{diferencia:.2f} punts", delta_color="inverse")

    st.markdown("---")
    st.subheader("🗺️ Distribució Geogràfica de la Vulnerabilitat")

    # Intentem pintar el mapa si hi ha coordenades
    if 'Latitud' in df_simulat.columns and 'Longitud' in df_simulat.columns:
        fig = px.scatter_mapbox(df_simulat, 
                                lat="Latitud", lon="Longitud", 
                                color="CVI_Simulat", 
                                hover_data=["CVI_Simulat"], # Pots afegir 'Seccio_Censal' si existeix
                                color_continuous_scale="RdYlGn_r", 
                                size_max=15, zoom=11, mapbox_style="carto-positron")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("💡 Com que l'arxiu actual no té columnes de Latitud i Longitud, es mostren els resultats en format llista (de barris més afectats a menys).")
        df_mostrar = df_simulat[['CVI_Predit_RF', 'CVI_Simulat']].copy()
        # Si tens la columna de la secció censal, afegeix-la:
        if 'Seccio_Censal' in df_simulat.columns:
            df_mostrar.insert(0, 'Seccio_Censal', df_simulat['Seccio_Censal'])
        
        st.dataframe(df_mostrar.sort_values(by='CVI_Simulat', ascending=False), use_container_width=True)

2026-06-05 19:12:06.597 
  command:

    streamlit run C:\Users\jordi\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-05 19:12:06.599 Session state does not function when running a script without `streamlit run`
2026-06-05 19:12:06.607 No runtime found, using MemoryCacheStorageManager
2026-06-05 19:12:08.731 No runtime found, using MemoryCacheStorageManager
